In [130]:
import pandas as pd

requests_df = pd.read_csv("./data/Восток Синтетические данные.csv", sep=';', encoding="windows-1251")

In [131]:
requests_df


,Заявка,Тип заявки BK,Тип заявки HD,Начало,Окончание,Район,Адрес,Подключение,Гигабитное подключение
0,74198,Подключение,Конвергенция абонента,17.08.2026 20:00,17.08.2026 22:00,Кузьминки,"Город Москва, пр-кт.Волгоградский, д. 128 к 5",FMC,Нет
1,86160,Подключение,Конвергенция абонента,17.08.2026 18:00,17.08.2026 20:00,Таганский,"Город Москва, пер.Маяковского, д. 2",FMC,Нет
2,50104,Подключение,Заявка на подключение,17.08.2026 18:00,17.08.2026 20:00,Текстильщики,"Город Москва, ул.Грайвороновская, д. 10 к 2",NaN,Да
3,46393,Подключение,Конвергенция абонента,17.08.2026 20:00,17.08.2026 22:00,Рязанский,"Город Москва, ул.Михайлова, д. 14",FMC,Нет
4,10135,Подключение,Конвергенция абонента,17.08.2026 18:00,17.08.2026 20:00,Рязанский,"Город Москва, ул.3-я Институтская, д. 5 к 2",FMC,Нет
...,...,...,...,...,...,...,...,...,...
64,15648,Глобальная проблема,Информация,17.08.2026 16:00,17.08.2026 18:00,Басманный,"г.Город Москва, наб.Семеновская, д. 2/1",NaN,Нет
65,57299,Глобальная проблема,Авария,17.08.2026 20:00,17.08.2026 22:00,Кузьминки,"г.Город Москва, пр-кт.Волгоградский, д. 128к1",NaN,Нет
66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
67,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [132]:
def get_engineers_info(control_df: pd.DataFrame) -> pd.DataFrame:
    _start = pd.to_datetime(control_df["Начало"], format="%d.%m.%Y %H:%M")
    _end = pd.to_datetime(control_df["Окончание"], format="%d.%m.%Y %H:%M")

    agg_kwargs = {
        "shift_start": ("_start", "min"),
        "shift_end": ("_end", "max"),
        "can_gigabit": ("Гигабитное подключение", lambda s: "Да" in set(s.dropna().unique())),
    }

    if "Подключение" in control_df.columns:
        agg_kwargs["equipment_types"] = (
            "Подключение",
            lambda s: set(s.dropna().unique()) or None
        )

    engineers_df = (
        control_df.assign(_start=_start, _end=_end)
        .dropna(subset=["Бригада"])
        .groupby("Бригада")
        .agg(**agg_kwargs)
        .reset_index()
        .sort_values("shift_start")
    )

    engineers_df["shift_start"] = engineers_df["shift_start"].dt.time
    engineers_df["shift_end"] = engineers_df["shift_end"].dt.time

    if "equipment_types" not in engineers_df.columns:
        engineers_df["equipment_types"] = [None for _ in range(len(engineers_df))]

    return engineers_df

In [133]:
east_control_df = pd.read_csv("./data/Восток Контрольное распределение..csv", sep=';', encoding="windows-1251")

In [134]:
engineers_east_df = get_engineers_info(east_control_df)
engineers_east_df["Office"] = "г. Москва, ул Юных Ленинцев, д 83с 4"

In [135]:
engineers_east_df.to_csv("./data/EastEngineersData.csv", index=False)

In [136]:
south_east_control_df = pd.read_csv("./data/Юго-восток Контрольное распределение.csv", sep=';', encoding="windows-1251")

In [137]:
south_east_engineers_df = get_engineers_info(south_east_control_df)
south_east_engineers_df["Office"] = "г. Москва, ул Бирюлёвская, д 1с1"
south_east_engineers_df.to_csv("./data/SouthEastEngineersData.csv", index=False)

In [138]:
south_east_control_df = pd.read_csv("./data/Югоцентр Контрольное распределение..csv", sep=';', encoding="windows-1251")

In [139]:
south_center_east_engineers_df = get_engineers_info(south_east_control_df)
south_center_east_engineers_df["Office"] = "г.Москва проезд Симферопольский, д.7"
south_center_east_engineers_df.to_csv("./data/SouthCenterEngineersData.csv", index=False)